# Cost-Aware FinQA — Training an LLM for Strategic Tool Selection

This notebook trains a small LLM (Qwen 2.5-1.5B) using **Unsloth** and **Expert Iteration** (rejection sampling + SFT) to learn cost-aware tool selection in financial question answering.

**Environment:** [Cost-Aware FinQA HF Space](https://huggingface.co/spaces/Teachafy/cost-aware-finqa)  
**Source:** [GitHub](https://github.com/nsharan2000/cost-aware-finqa)

## What the agent learns

The agent has 5 tools with different costs:
| Tool | Cost | When to use |
|------|------|-------------|
| `sql_query` | FREE | Numerical lookups from financial tables |
| `vector_search` | $0.50 | Context from SEC filing text |
| `web_search` | $3.00 | Industry benchmarks / external data |
| `upgrade_llm` | $3.00 | Complex multi-step reasoning |
| `submit_answer` | FREE | Submit final answer |

**Scoring:** `correctness × cost_efficiency × (1 - error_penalty)`  
A correct answer with $0 cost scores 1.0; the same answer after $3 on web search scores ~0.7.

## Phases
1. **Setup** — Install dependencies, clone environment, load model
2. **Random baseline** — Measure performance of random tool selection
3. **Pre-train LLM baseline** — Qwen 2.5-1.5B out-of-the-box
4. **Expert Iteration training** — Best-of-N rejection sampling + SFT
5. **Post-train evaluation** — Measure improvement
6. **Comparison plots** — Visualize results

## Cell 1 — Install Dependencies

In [ ]:
!pip install -q unsloth trl datasets transformers accelerate
!pip install -q pydantic requests matplotlib numpy

## Cell 2 — Clone Environment from GitHub & Verify HF Space

In [ ]:
import os, sys, json, requests

# ── Clone the environment code from GitHub ──────────────────────────────────
REPO_DIR = "/content/cost_aware_finqa"
if not os.path.exists(REPO_DIR):
    os.system(f"git clone https://github.com/nsharan2000/cost-aware-finqa.git {REPO_DIR}")
    print(f"Cloned environment code to {REPO_DIR}")
else:
    print(f"Code already at {REPO_DIR}")

# Add to path so we can import the environment locally
sys.path.insert(0, REPO_DIR)

# ── Verify the deployed HF Space is live ───────────────────────────────────
SPACE_URL = "https://Teachafy-cost-aware-finqa.hf.space"

try:
    resp = requests.post(f"{SPACE_URL}/reset", json={}, timeout=15)
    obs = resp.json().get("observation", {})
    print(f"HF Space is live at {SPACE_URL}")
    print(f"  Question: {obs.get('question', '')[:80]}...")
    print(f"  Budget: ${obs.get('budget_total', 0):.2f}")
    print(f"  Tools: {obs.get('available_tools', [])}")
except Exception as e:
    print(f"Space not reachable ({e}). Training will use local env only.")

## Cell 3 — Import Local Environment & Sanity Check

In [ ]:
from server.cost_aware_finqa_environment import CostAwareFinqaEnvironment, TASK_CONFIG
from models import CostAwareFinqaAction
from server.tools import TOOL_COSTS

print(f"Environment imported successfully")
print(f"Tasks: {list(TASK_CONFIG.keys())}")
print(f"Tool costs: {TOOL_COSTS}")

# Quick sanity check
env = CostAwareFinqaEnvironment()
obs = env.reset()
print(f"\nLocal reset OK")
print(f"  Question: {obs.question[:80]}...")
print(f"  Budget: ${obs.budget_total:.2f}")
print(f"  Max steps: {obs.max_steps}")
print(f"  Table schema: {obs.table_schema[:120]}...")

## Cell 4 — Load Qwen 2.5-1.5B with Unsloth (4-bit, LoRA)

In [ ]:
from unsloth import FastLanguageModel
import torch

MODEL_NAME  = "unsloth/Qwen2.5-1.5B-Instruct-bnb-4bit"
MAX_SEQ_LEN = 2048
LORA_R      = 16

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=MAX_SEQ_LEN,
    load_in_4bit=True,
)

model = FastLanguageModel.get_peft_model(
    model,
    r=LORA_R,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                     "gate_proj", "up_proj", "down_proj"],
    lora_alpha=16,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
)

print(f"Model loaded: {MODEL_NAME}")
model.print_trainable_parameters()

In [ ]:
# Enable fast inference mode (Unsloth optimization)
FastLanguageModel.for_inference(model)

## Cell 5 — Prompt Engineering & Action Parsing

In [ ]:
import re, random, textwrap
import numpy as np
from collections import defaultdict

# ── Tool descriptions for the prompt ───────────────────────────────────────
TOOL_DESCS = {
    "sql_query":      "Run SQL on financial tables. FREE. Use for numerical lookups. Penalized for bad SQL.",
    "vector_search":  "Search SEC filing text. $0.50. Use when you need narrative context.",
    "web_search":     "External web search. $3.00. Only for industry benchmarks/comparisons.",
    "upgrade_llm":    "Stronger model for reasoning. $3.00. Only for complex multi-step calculations.",
    "submit_answer":  "Submit final answer. FREE. Answer should be a number or short phrase.",
}

SYSTEM_PROMPT = textwrap.dedent("""\
You are a cost-aware financial research agent. You answer financial questions by
choosing the cheapest tool that gets the job done.

Available tools:
{tool_lines}

Strategy:
- ALWAYS try sql_query first (it's free).
- Use 'SELECT * FROM table_catalog' to discover available tables.
- Only use expensive tools when sql_query can't answer the question.
- Minimize total cost to maximize your score.

Scoring: correctness * (1 - cost/budget) * (1 - error_penalty)
Bad SQL queries give -0.15 penalty each. Redundant calls give -0.05.

Respond with ONLY a JSON object:
{{"tool": "<tool_name>", "query": "<your query>", "answer": "<if submitting>"}}
""")


def build_chat_messages(obs, last_result=None):
    """Build chat messages in Qwen's expected format."""
    tool_lines = "\n".join(f"  - {name}: {desc}" for name, desc in TOOL_DESCS.items())
    system = SYSTEM_PROMPT.format(tool_lines=tool_lines)

    user_parts = [
        f"Question: {obs.question}",
        f"Budget: ${obs.budget_remaining:.2f} / ${obs.budget_total:.2f}",
        f"Step: {obs.step_number} / {obs.max_steps}",
    ]

    if obs.table_schema:
        user_parts.append(f"Table Schema:\n{obs.table_schema[:400]}")

    if last_result:
        user_parts.append(f"Last tool result:\n{last_result[:300]}")

    if obs.error:
        user_parts.append(f"Error: {obs.error}")

    return [
        {"role": "system", "content": system},
        {"role": "user", "content": "\n\n".join(user_parts)},
    ]


def parse_action(text):
    """Parse LLM output into (tool, query, answer)."""
    text = text.strip()
    try:
        # Try to extract JSON
        if "{" in text:
            start = text.index("{")
            end = text.rindex("}") + 1
            blob = json.loads(text[start:end])
            return (
                blob.get("tool", "submit_answer"),
                blob.get("query", ""),
                blob.get("answer", ""),
            )
    except (json.JSONDecodeError, ValueError):
        pass
    # Fallback: submit the raw text as answer
    return ("submit_answer", "", text[:100])


def generate_action(model, tokenizer, obs, last_result=None, temperature=0.7):
    """Generate a single action from the model."""
    messages = build_chat_messages(obs, last_result)
    prompt = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True,
    )

    inputs = tokenizer(
        prompt, return_tensors="pt",
        truncation=True, max_length=MAX_SEQ_LEN,
    ).to(model.device)

    with torch.no_grad():
        ids = model.generate(
            **inputs, max_new_tokens=128,
            temperature=temperature,
            do_sample=True, top_p=0.95,
        )

    gen_text = tokenizer.decode(ids[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
    tool, query, answer = parse_action(gen_text)
    return gen_text, {"tool": tool, "query": query, "answer": answer}

print("Prompt engineering & action parsing ready.")

## Cell 6 — Reward Shaping (Dense Signal for Training)

In [ ]:
# Per-tool step rewards (mirrors the environment's reward structure)
TOOL_STEP_REWARDS = {
    "sql_query":      0.05,   # Free and usually the right first step
    "vector_search":  0.03,   # Useful but costs money
    "web_search":    -0.02,   # Expensive, usually unnecessary
    "upgrade_llm":   -0.02,   # Expensive, usually unnecessary
    "submit_answer":  0.00,   # Neutral (final score matters)
}


def shaped_reward(gen_text, action, obs):
    """Dense reward signal for training.

    Components:
      - Format bonus: +0.10 if valid JSON with tool field
      - Tool choice: reward from TOOL_STEP_REWARDS
      - Cost awareness: bonus for using free tools first
      - Error: penalty if the tool result had an error
    """
    reward = 0.0

    # Format bonus
    try:
        blob = json.loads(gen_text.strip()) if "{" in gen_text else None
        if blob and isinstance(blob, dict) and "tool" in blob:
            reward += 0.10
            if "query" in blob or "answer" in blob:
                reward += 0.05
    except (json.JSONDecodeError, ValueError):
        reward -= 0.10  # Unparseable output

    # Tool choice reward
    tool = action.get("tool", "")
    reward += TOOL_STEP_REWARDS.get(tool, 0.0)

    # Cost awareness: bonus for using free tools early in episode
    if tool == "sql_query" and obs.step_number <= 2:
        reward += 0.05  # Good: exploring with free tool first

    # Penalty for using expensive tools when budget is tight
    tool_cost = TOOL_COSTS.get(tool, 0.0)
    if tool_cost > 0 and obs.budget_remaining < obs.budget_total * 0.5:
        reward -= 0.05  # Wasteful spending

    # Error penalty
    if obs.error:
        reward -= 0.10

    return reward

print("Reward shaping ready.")

## Cell 7 — Episode Runner

In [ ]:
def run_episode(
    model,
    tokenizer,
    task: str = "basic_retrieval",
    seed: int = None,
    max_steps: int = 8,
    temperature: float = 0.7,
    verbose: bool = False,
) -> dict:
    """Run a full multi-step episode, return trajectory + rewards."""
    ep_seed = seed or random.randint(0, 999_999)

    os.environ["FINQA_TASK"] = task
    env = CostAwareFinqaEnvironment()

    # Reset to a specific question (deterministic via seed)
    random.seed(ep_seed)
    env._question_index = ep_seed % max(len(env._task_questions) if env._task_questions else 1, 1)
    obs = env.reset()

    trajectory = []
    total_env_reward = 0.0
    total_shaped = 0.0
    last_result = None
    tools_used = set()

    for step_i in range(max_steps):
        gen_text, action = generate_action(
            model, tokenizer, obs, last_result, temperature=temperature
        )

        # Step environment
        act = CostAwareFinqaAction(
            tool=action["tool"],
            query=action["query"],
            answer=action["answer"],
        )
        obs = env.step(act)
        env_reward = obs.reward
        total_env_reward += env_reward
        last_result = obs.tool_result
        tools_used.add(action["tool"])

        # Shaped reward
        sr = shaped_reward(gen_text, action, obs)
        total_shaped += sr

        trajectory.append({
            "step": step_i,
            "action": action,
            "gen_text": gen_text[:200],
            "env_reward": env_reward,
            "shaped_rw": sr,
            "tool_cost": obs.tool_cost,
        })

        if verbose:
            cost_str = f"${obs.tool_cost:.2f}" if obs.tool_cost > 0 else "FREE"
            print(f"  [{step_i:02d}] {action['tool']:<16s} ({cost_str})  "
                  f"env={env_reward:+.4f}  shaped={sr:+.4f}  "
                  f"gen='{gen_text[:55]}...'")

        if obs.done:
            break

    return {
        "trajectory":     trajectory,
        "total_reward":   total_env_reward,
        "total_shaped":   total_shaped,
        "final_score":    obs.score,
        "cost_spent":     obs.cost_so_far,
        "budget_total":   obs.budget_total,
        "steps":          len(trajectory),
        "tools_used":     len(tools_used),
        "tools":          tools_used,
        "done":           obs.done,
        "question":       obs.question[:80],
        "task":           task,
    }

print("Episode runner ready.")

## Cell 8 — Phase 1: Random Agent Baseline

In [ ]:
print("=" * 60)
print("Phase 1 / 5 \u2014 Random Agent Baseline")
print("=" * 60)

NUM_RANDOM = 10
random_scores = []
random_costs = []

VALID_TOOLS = ["sql_query", "vector_search", "web_search", "upgrade_llm", "submit_answer"]

for i in range(NUM_RANDOM):
    os.environ["FINQA_TASK"] = random.choice(list(TASK_CONFIG.keys()))
    env = CostAwareFinqaEnvironment()
    obs = env.reset()

    for _ in range(obs.max_steps):
        tool = random.choice(VALID_TOOLS)
        query = ""
        answer = ""
        if tool == "sql_query":
            query = random.choice([
                "SELECT * FROM table_catalog LIMIT 5",
                "SELECT * FROM table_catalog",
                "SELECT name FROM sqlite_master WHERE type='table'",
            ])
        elif tool == "vector_search":
            query = "financial performance revenue"
        elif tool == "web_search":
            query = "company financial benchmarks"
        elif tool == "submit_answer":
            answer = str(random.uniform(0, 1))

        act = CostAwareFinqaAction(tool=tool, query=query, answer=answer)
        obs = env.step(act)
        if obs.done:
            break

    random_scores.append(obs.score)
    random_costs.append(obs.cost_so_far)
    print(f"  Random ep {i+1:02d}: score={obs.score:.4f}  cost=${obs.cost_so_far:.2f}")

print(f"  >> Random baseline: {np.mean(random_scores):.4f} +/- {np.std(random_scores):.4f}")
print(f"  >> Avg cost: ${np.mean(random_costs):.2f}")

## Cell 9 — Phase 2: Pre-Training LLM Baseline

In [ ]:
print("\n" + "=" * 60)
print("Phase 2 / 5 \u2014 Pre-Training LLM Baseline")
print("=" * 60)

NUM_EVAL = 8
SEEDS = [i * 137 for i in range(NUM_EVAL)]
EVAL_TASKS = ["basic_retrieval", "basic_retrieval", "analytical_reasoning",
              "analytical_reasoning", "strategic_research", "strategic_research",
              "basic_retrieval", "analytical_reasoning"]

baseline_scores = []
baseline_shaped = []
baseline_costs = []
baseline_steps = []

for i, (seed, task) in enumerate(zip(SEEDS, EVAL_TASKS)):
    res = run_episode(model, tokenizer, task=task, seed=seed,
                      verbose=(i == 0))  # Show first episode in detail
    baseline_scores.append(res["final_score"])
    baseline_shaped.append(res["total_shaped"])
    baseline_costs.append(res["cost_spent"])
    baseline_steps.append(res["steps"])
    print(f"  Ep {i+1:02d} [{task[:8]}]: score={res['final_score']:.4f}  "
          f"shaped={res['total_shaped']:+.4f}  "
          f"cost=${res['cost_spent']:.2f}  "
          f"steps={res['steps']}  tools={res['tools']}")

print(f"  >> Pre-train score mean:  {np.mean(baseline_scores):.4f} +/- {np.std(baseline_scores):.4f}")
print(f"  >> Pre-train shaped mean: {np.mean(baseline_shaped):.4f} +/- {np.std(baseline_shaped):.4f}")
print(f"  >> Pre-train avg cost:    ${np.mean(baseline_costs):.2f}")

## Cell 10 — Phase 3: Expert Iteration Training

Uses rejection sampling (Best-of-N) to generate good trajectories, then trains with SFT.
This is more stable than GRPO and avoids Unsloth tensor compatibility issues.

In [ ]:
from datasets import Dataset
from trl import SFTTrainer, SFTConfig

# Ensure pad token is set
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

print("\n" + "=" * 60)
print("Phase 3 / 5 \u2014 Expert Iteration Training")
print("=" * 60)
print("(Rejection Sampling + SFT)")

# ── Config ─────────────────────────────────────────────────────────────────
NUM_PROMPTS = 24        # distinct episode seeds
N_SAMPLES   = 4         # completions per prompt (keep the best)
NUM_ROUNDS  = 3         # expert-iteration rounds
TRAIN_TASKS = ["basic_retrieval", "analytical_reasoning", "strategic_research"]

round_mean_scores = []  # track improvement across rounds
train_losses_all  = []  # for plotting

for rnd in range(NUM_ROUNDS):
    print(f"\n  --- Round {rnd+1}/{NUM_ROUNDS} ---")

    # ── 1. Generate completions & keep best-of-N per prompt ────────────────
    best_pairs = []
    round_best_scores = []

    model = FastLanguageModel.for_inference(model)

    for j in range(NUM_PROMPTS):
        seed = j * 53 + 7 + rnd * 1000
        task = TRAIN_TASKS[j % len(TRAIN_TASKS)]

        os.environ["FINQA_TASK"] = task
        env = CostAwareFinqaEnvironment()
        random.seed(seed)
        env._question_index = seed % max(len(env._task_questions) if env._task_questions else 1, 1)
        obs = env.reset()

        messages = build_chat_messages(obs)
        prompt_text = tokenizer.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=True,
        )

        best_score = -999.0
        best_completion = None

        for k in range(N_SAMPLES):
            inputs = tokenizer(
                prompt_text, return_tensors="pt",
                truncation=True, max_length=MAX_SEQ_LEN,
            ).to(model.device)

            with torch.no_grad():
                ids = model.generate(
                    **inputs, max_new_tokens=128,
                    temperature=0.8 + 0.1 * k,
                    do_sample=True, top_p=0.95,
                )
            completion = tokenizer.decode(
                ids[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True
            )

            # Score this completion by running it in the env
            tool, query, answer = parse_action(completion)

            # Reset env for scoring
            score_env = CostAwareFinqaEnvironment()
            random.seed(seed)
            score_env._question_index = seed % max(len(score_env._task_questions) if score_env._task_questions else 1, 1)
            score_obs = score_env.reset()

            # Run the action
            act = CostAwareFinqaAction(tool=tool, query=query, answer=answer)
            score_obs = score_env.step(act)
            sr = shaped_reward(completion, {"tool": tool, "query": query, "answer": answer}, score_obs)

            if sr > best_score:
                best_score = sr
                best_completion = completion

        if best_completion:
            full_text = prompt_text + best_completion
            best_pairs.append({"text": full_text})
            round_best_scores.append(best_score)

    round_mean = np.mean(round_best_scores) if round_best_scores else 0.0
    round_mean_scores.append(round_mean)
    print(f"  Generated {len(best_pairs)} training examples")
    print(f"  Best-of-{N_SAMPLES} mean shaped reward: {round_mean:.4f}")

    # ── 2. SFT on the best completions ─────────────────────────────────────
    ds = Dataset.from_list(best_pairs)

    model = FastLanguageModel.for_training(model)

    trainer = SFTTrainer(
        model=model,
        train_dataset=ds,
        args=SFTConfig(
            output_dir=f"./finqa_sft_round{rnd+1}",
            per_device_train_batch_size=2,
            gradient_accumulation_steps=4,
            num_train_epochs=2,
            learning_rate=2e-4,
            warmup_steps=5,
            fp16=not torch.cuda.is_bf16_supported(),
            bf16=torch.cuda.is_bf16_supported(),
            logging_steps=1,
            max_seq_length=MAX_SEQ_LEN,
            dataset_text_field="text",
        ),
        processing_class=tokenizer,
    )

    result = trainer.train()
    train_losses_all.append(result.training_loss)
    print(f"  Round {rnd+1} training loss: {result.training_loss:.4f}")

print(f"\nTraining complete! Round scores: {[f'{s:.4f}' for s in round_mean_scores]}")
print(f"Training losses: {[f'{l:.4f}' for l in train_losses_all]}")

## Cell 11 — Phase 4: Post-Training Evaluation

In [ ]:
print("\n" + "=" * 60)
print("Phase 4 / 5 \u2014 Post-Training Evaluation")
print("=" * 60)

model = FastLanguageModel.for_inference(model)

post_scores = []
post_shaped = []
post_costs  = []
post_steps  = []

for i, (seed, task) in enumerate(zip(SEEDS, EVAL_TASKS)):
    res = run_episode(model, tokenizer, task=task, seed=seed,
                      verbose=(i == 0))
    post_scores.append(res["final_score"])
    post_shaped.append(res["total_shaped"])
    post_costs.append(res["cost_spent"])
    post_steps.append(res["steps"])
    print(f"  Ep {i+1:02d} [{task[:8]}]: score={res['final_score']:.4f}  "
          f"shaped={res['total_shaped']:+.4f}  "
          f"cost=${res['cost_spent']:.2f}  "
          f"steps={res['steps']}  tools={res['tools']}")

print(f"  >> Post-train score mean:  {np.mean(post_scores):.4f} +/- {np.std(post_scores):.4f}")
print(f"  >> Post-train shaped mean: {np.mean(post_shaped):.4f} +/- {np.std(post_shaped):.4f}")
print(f"  >> Post-train avg cost:    ${np.mean(post_costs):.2f}")
print(f"  >> Score improvement:      {np.mean(post_scores) - np.mean(baseline_scores):+.4f}")
print(f"  >> Shaped improvement:     {np.mean(post_shaped) - np.mean(baseline_shaped):+.4f}")
print(f"  >> Cost reduction:         ${np.mean(baseline_costs) - np.mean(post_costs):+.2f}")

## Cell 12 — Phase 5: Comparison Plots

In [ ]:
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams.update({"font.size": 12})

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# ── 1. Bar chart: Score comparison ──────────────────────────────────────────
labels = ["Random\nAgent", "Pre-Train\nLLM", "Post-Train\nLLM"]
means  = [np.mean(random_scores), np.mean(baseline_scores), np.mean(post_scores)]
stds   = [np.std(random_scores),  np.std(baseline_scores),  np.std(post_scores)]
colors = ["#95a5a6", "#e74c3c", "#2ecc71"]

bars = axes[0].bar(labels, means, yerr=stds, color=colors, capsize=8,
                   edgecolor="black", linewidth=0.8)
axes[0].set_ylabel("Mean Score")
axes[0].set_title("Cost-Aware FinQA \u2014 Agent Comparison")
axes[0].grid(axis="y", alpha=0.3)

if means[2] > means[1]:
    delta = means[2] - means[1]
    pct = (delta / max(abs(means[1]), 0.001)) * 100
    axes[0].annotate(f"+{pct:.0f}%\n({delta:+.3f})",
                     xy=(2, means[2] + stds[2] + 0.02),
                     fontsize=11, fontweight="bold", color="#27ae60",
                     ha="center", va="bottom")

# ── 2. Per-episode comparison ──────────────────────────────────────────────
x = range(1, NUM_EVAL + 1)
axes[1].plot(x, baseline_scores, "o-", color="#e74c3c", linewidth=2,
             markersize=7, label="Pre-train")
axes[1].plot(x, post_scores, "s-", color="#2ecc71", linewidth=2,
             markersize=7, label="Post-train")
axes[1].fill_between(x, baseline_scores, post_scores,
                     alpha=0.15, color="#2ecc71",
                     where=[p > b for p, b in zip(post_scores, baseline_scores)])
axes[1].set_xlabel("Episode")
axes[1].set_ylabel("Final Score")
axes[1].set_title("Per-Episode Score")
axes[1].legend(loc="lower right")
axes[1].grid(alpha=0.3)

# ── 3. Cost comparison ─────────────────────────────────────────────────────
cost_labels = ["Random", "Pre-Train", "Post-Train"]
cost_means = [np.mean(random_costs), np.mean(baseline_costs), np.mean(post_costs)]
cost_colors = ["#95a5a6", "#e74c3c", "#2ecc71"]

axes[2].bar(cost_labels, cost_means, color=cost_colors,
            edgecolor="black", linewidth=0.8)
axes[2].set_ylabel("Mean Cost ($)")
axes[2].set_title("Average Episode Cost")
axes[2].grid(axis="y", alpha=0.3)

# Add cost values on bars
for bar, val in zip(axes[2].patches, cost_means):
    axes[2].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.1,
                f"${val:.2f}", ha="center", fontweight="bold")

plt.tight_layout()
plt.savefig("finqa_training_results.png", dpi=150, bbox_inches="tight")
plt.show()
print("Plot saved to finqa_training_results.png")

## Cell 13 — Detailed Episode Walkthrough

In [ ]:
print("\n" + "=" * 60)
print("Detailed Episode Walkthrough (post-training)")
print("=" * 60)

# Run one detailed episode per task
for task in ["basic_retrieval", "analytical_reasoning", "strategic_research"]:
    print(f"\n--- Task: {task} ---")
    res = run_episode(model, tokenizer, task=task, seed=42, verbose=True)
    print(f"  Final score: {res['final_score']:.4f}")
    print(f"  Cost spent: ${res['cost_spent']:.2f} / ${res['budget_total']:.2f}")
    print(f"  Tools used: {res['tools']}")
    print(f"  Question: {res['question']}")

## Cell 14 — Final Summary

In [ ]:
print("\n" + "=" * 70)
print("  COST-AWARE FINQA \u2014 TRAINING RESULTS")
print("=" * 70)
print(f"  Environment:   Cost-Aware FinQA")
print(f"  HF Space:      {SPACE_URL}")
print(f"  Model:         {MODEL_NAME}")
print(f"  LoRA rank:     {LORA_R}")
print(f"  Training:      Expert Iteration (Best-of-{N_SAMPLES} + SFT x {NUM_ROUNDS} rounds)")
print(f"                 {NUM_PROMPTS} prompts per round")
print(f"  " + "-" * 50)
print(f"  {'Metric':<22s} {'Score':>10s}  {'Shaped':>10s}  {'Avg Cost':>10s}")
print(f"  {'-'*22} {'-'*10}  {'-'*10}  {'-'*10}")
print(f"  {'Random agent':<22s} {np.mean(random_scores):>+10.4f}  {'N/A':>10s}  ${np.mean(random_costs):>8.2f}")
print(f"  {'Pre-train LLM':<22s} {np.mean(baseline_scores):>+10.4f}  {np.mean(baseline_shaped):>+10.4f}  ${np.mean(baseline_costs):>8.2f}")
print(f"  {'Post-train LLM':<22s} {np.mean(post_scores):>+10.4f}  {np.mean(post_shaped):>+10.4f}  ${np.mean(post_costs):>8.2f}")
score_imp = np.mean(post_scores) - np.mean(baseline_scores)
score_pct = (score_imp / max(abs(np.mean(baseline_scores)), 0.001)) * 100
print(f"  " + "-" * 50)
print(f"  Score delta:    {score_imp:+.4f}  ({score_pct:+.0f}%)")
print(f"  Round scores:   {[f'{s:.4f}' for s in round_mean_scores]}")
print(f"  Round losses:   {[f'{l:.4f}' for l in train_losses_all]}")
print("=" * 70)
print("\nKey insight: The trained agent learns to prefer free SQL queries")
print("over expensive tools, reducing cost while maintaining accuracy.")

---

## Next Steps

1. **Scale up**: Increase `NUM_PROMPTS` and `NUM_ROUNDS` for better results
2. **Multi-step training**: Train on full episode trajectories instead of single steps
3. **GRPO**: Once Unsloth GRPO tensor compatibility is resolved, switch to direct RL
4. **Deploy**: Push the LoRA adapter to HuggingFace Hub and use it in `inference.py`

### Resources
- [Cost-Aware FinQA HF Space](https://huggingface.co/spaces/Teachafy/cost-aware-finqa)
- [GitHub Repository](https://github.com/nsharan2000/cost-aware-finqa)
- [Unsloth Documentation](https://docs.unsloth.ai/)
- [TRL (Transformer Reinforcement Learning)](https://huggingface.co/docs/trl/)